In [1]:
import os 

In [2]:
%pwd

'd:\\PredictBot-Score-MLOps\\research'

In [3]:

# ".." tells Python to step out of the current folder into the parent folder
os.chdir("..")

# Check where you are now


In [4]:
%pwd


'd:\\PredictBot-Score-MLOps'

In [5]:
from src.predictor_bot_score.constants import CONFIG_PATH
from src.predictor_bot_score.logger import logger
from src.predictor_bot_score.utils.common import yaml_load , create_directories
from dataclasses import dataclass
from pathlib import Path
import os
import boto3
from botocore.exceptions import (
    ClientError,
    NoCredentialsError,
    PartialCredentialsError,
    EndpointConnectionError,
    ConnectTimeoutError,
)



In [6]:
@dataclass(frozen=True)
class Data_injestion_config:

    bucket_name : str 
    file_name  : str
    raw_data : Path
    ingested_data : Path



In [7]:
class yaml_configruation:

    def __init__(self,config_path = CONFIG_PATH):
        self.config_path = yaml_load(config_path)

        create_directories([self.config_path.artifacts_root])

    def get_data_ingestion_config(self)-> Data_injestion_config:

        config_path = self.config_path.data_ingestion

        create_directories([config_path.raw_data,config_path.ingested_data])

        return Data_injestion_config(
            bucket_name = config_path.bucket_name,
            file_name = config_path.file_name,
            raw_data = config_path.raw_data,
            ingested_data = config_path.ingested_data

        )


In [ ]:
class Data_injestion:

    def __init__(self, config : Data_injestion_config):

        self.config = config
        self.s3 =   boto3.client(
                        's3', 
                        aws_access_key_id= os.getenv("AWS_ACCESS_KEY_ID"),
                        aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"))
        self.BUCKET_NAME = self.config.bucket_name
        self.FILE_NAME = self.config.file_name

    def connection_check(self):

        try:

            response = self.s3.list_objects_v2(Bucket=self.BUCKET_NAME)
            return True

        except ClientError as c:
            # Extract the specific AWS error code dictionary string
            error_code = c.response['Error']['Message']
            print( error_code)
            raise
            
    
    def file_check(self):

        try:
            if self.connection_check() :
                logger.info("S3 . Connection exists ")
                response  = self.s3.list_objects_v2(Bucket = self.BUCKET_NAME)
                FIlES =([i['Key'] for i in response['Contents']])

                if self.FILE_NAME in FIlES:
                    logger.info(f"{self.FILE_NAME} FILE EXISTS")
                    return self.FILE_NAME
                else: 
                    raise FileNotFoundError(
                        f"'{self.FILE_NAME}' does not exist in bucket '{self.BUCKET_NAME}'")


        except ClientError as e:
            logger.error(f"S3 connection error: {e.response['Error']['Message']}")
            raise

    def s3_downlaod_data(self):
    
        try:
            s3_file_name =  self.file_check() 
            file_name = os.path.basename(s3_file_name)
            save_path = os.path.join(self.config.raw_data,file_name)

            if os.path.exists(save_path):
                logger.info(f"{save_path}: {self.FILE_NAME}Path exists ")

            else:
                content = self.s3.download_file(
                        Bucket=self.BUCKET_NAME,
                        Key=self.FILE_NAME,
                        Filename=str(save_path)
                    )
                logger.info(f"{self.FILE_NAME} created at path {save_path}")

        except FileNotFoundError as e:
            logger.error(f"Download failed — file not found: {e}")
            raise
        except ClientError as e:
            logger.error(f"Download failed — AWS error: {e.response['Error']['Message']}")
            raise


            




In [ ]:
def connection_ckeck():

    try:
        s3 =   boto3.client(
                    's3', 
                    aws_access_key_id='REMOVED_AWS_KEY',
                    aws_secret_access_key='REMOVED_AWS_SECRET')
        
        response = s3.list_objects_v2(Bucket='predict-bot-master-data')
        return True

    except ClientError as c:
        # Extract the specific AWS error code dictionary string
        error_code = c.response['Error']['Message']
        print( error_code)
        raise
        
    

In [ ]:
connection_ckeck()

True

In [12]:
con = yaml_configruation()
con_data_injestion = con.get_data_ingestion_config()
con_data_injestion = Data_injestion(con_data_injestion)
con_data_injestion.s3_downlaod_data()


[2026-05-30 13:16:14,761: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-30 13:16:14,763: INFO: common: logger.info(f"created directory at: artifacts")]
[2026-05-30 13:16:14,767: INFO: common: logger.info(f"created directory at: artifacts/data_ingestion/ingested_data")]
[2026-05-30 13:16:25,151: INFO: 1046587706: S3 . Connection exists ]
[2026-05-30 13:16:27,067: INFO: 1046587706: Cloud_fare_Master_data.csv FILE EXISTS]
[2026-05-30 13:16:37,348: INFO: 1046587706: Cloud_fare_Master_data.csv created at path artifacts/data_ingestion/raw_data\Cloud_fare_Master_data.csv]
